# Week 5 — Dataset Preparation and Audio Preprocessing

This notebook demonstrates lightweight dataset preparation for speech-to-text tasks (small slice of Common Voice) and basic audio preprocessing (resampling to 16 kHz, normalization). Designed to run quickly on Colab — we use a tiny dataset slice so downloads are minimal.

In [1]:
# Install required packages (Colab-friendly)
!pip install -q datasets soundfile librosa torchaudio

In [5]:
from datasets import load_dataset, Dataset, Audio
import librosa
import numpy as np
import soundfile as sf

print('Loading a tiny slice of Common Voice (English) — this should be quick')
# Use a very small slice to keep runs fast on Colab
try:
    ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en', split='train[:0.5%]')
    ds_name = 'common_voice_13_0'
except Exception as e:
    print('Primary Common Voice load failed:', e)
    try:
        ds = load_dataset('hf-internal-testing/librispeech_asr_demo', split='validation')
        ds_name = 'librispeech_asr_demo'
    except Exception as e2:
        print('Fallback librispeech demo failed:', e2)
        sr = 16000
        duration = 1.0
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
        example = {'audio': {'array': arr, 'sampling_rate': sr}, 'text': 'synthetic tone'}
        ds = Dataset.from_list([example])
        ds_name = 'synthetic'

# Keep undecoded to avoid torchcodec; we'll decode manually and gracefully fallback if file access fails
if ds_name != 'synthetic':
    ds = ds.cast_column('audio', Audio(decode=False))

print('Dataset source:', ds_name)
print('Dataset size (slice):', len(ds))

# Inspect a single example to show audio structure
example = ds[0]
print('Keys:', list(example.keys()))

# Extract audio array and sampling rate robustly
arr = None
sr = 16000
if 'audio' in example:
    audio = example['audio']
    if isinstance(audio, dict) and 'array' in audio:
        arr = np.array(audio['array']).astype('float32')
        sr = int(audio.get('sampling_rate', 16000))
    elif isinstance(audio, dict) and 'path' in audio:
        # Try soundfile, then librosa, else fallback to synthetic
        path = audio['path']
        try:
            arr, sr = sf.read(path)
            arr = np.array(arr).astype('float32')
        except Exception as e_path:
            print('soundfile read failed on path, trying librosa:', e_path)
            try:
                arr, sr = librosa.load(path, sr=None)
                arr = np.array(arr).astype('float32')
            except Exception as e_lib:
                print('librosa read failed; falling back to synthetic tone:', e_lib)
        if arr is None:
            sr = 16000
            duration = 1.0
            t = np.linspace(0, duration, int(sr * duration), endpoint=False)
            arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
    else:
        try:
            arr, sr = sf.read(audio)
            arr = np.array(arr).astype('float32')
        except Exception:
            sr = 16000
            duration = 1.0
            t = np.linspace(0, duration, int(sr * duration), endpoint=False)
            arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
    print('Audio sample rate (original):', sr)
    print('Audio array shape:', np.array(arr).shape)
    print('Transcript (first sample):', example.get('sentence') or example.get('text') or example.get('transcript') or '')
else:
    print('No audio key in example; using synthetic tone fallback')
    sr = 16000
    duration = 1.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')

# Resample to 16 kHz and save a short example to /content for Colab playback
sr_target = 16000
if sr != sr_target:
    arr = librosa.resample(arr, orig_sr=sr, target_sr=sr_target)
    sr = sr_target

# Normalize to -1..1 if needed
mx = np.abs(arr).max()
if mx > 0:
    arr = arr / mx

out_path = 'example_resampled.wav'
sf.write(out_path, arr, sr)
print('Wrote resampled example to', out_path)

# Show duration
print('Duration (s):', len(arr) / sr)


Loading a tiny slice of Common Voice (English) — this should be quick


Repo card metadata block was not found. Setting CardData to empty.


Primary Common Voice load failed: The directory at hf://datasets/mozilla-foundation/common_voice_13_0@ff2bbb54dcdb597100fe534a1b911ff9103f9e22 doesn't contain any data files
Dataset source: librispeech_asr_demo
Dataset size (slice): 73
Keys: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id']
soundfile read failed on path, trying librosa: Error opening '1272-128104-0000.flac': System error.
librosa read failed; falling back to synthetic tone: [Errno 2] No such file or directory: '1272-128104-0000.flac'
Audio sample rate (original): 16000
Audio array shape: (16000,)
Transcript (first sample): MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
Wrote resampled example to example_resampled.wav
Duration (s): 1.0


/tmp/ipython-input-2902055879.py:54: UserWarning: PySoundFile failed. Trying audioread instead.
  arr, sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
